In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
   import os
   for root, dirs, files in os.walk("/kaggle/input"):
       if root.count(os.sep) <= 4:
           print(root, "| subfolders:", dirs)

/kaggle/input | subfolders: ['datasets']
/kaggle/input/datasets | subfolders: ['tawsifurrahman']
/kaggle/input/datasets/tawsifurrahman | subfolders: ['covid19-radiography-database']


In [2]:
import os
BASE = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database"
print(os.listdir(BASE))

for name in os.listdir(BASE):
    p = os.path.join(BASE, name)
    if os.path.isdir(p):
        print(name, "->", os.listdir(p))

['COVID-19_Radiography_Dataset']
COVID-19_Radiography_Dataset -> ['Lung_Opacity.metadata.xlsx', 'Normal.metadata.xlsx', 'README.md.txt', 'COVID.metadata.xlsx', 'Normal', 'Lung_Opacity', 'Viral Pneumonia.metadata.xlsx', 'Viral Pneumonia', 'COVID']


In [3]:
import os
DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"

counts = {}
for cls in ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]:
    img_dir = os.path.join(DATA_DIR, cls, "images")
    counts[cls] = len(os.listdir(img_dir))
    print(cls, "->", counts[cls])

print("Total:", sum(counts.values()))

COVID -> 3616
Lung_Opacity -> 6012
Normal -> 10192
Viral Pneumonia -> 1345
Total: 21165


In [4]:
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [5]:
import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"

FOLDERS = {
    "COVID": "COVID",
    "Lung_Opacity": "Lung_Opacity",
    "Normal": "Normal",
    "Viral Pneumonia": "Viral_Pneumonia",
}
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral_Pneumonia"]

rows = []
for folder, cls in FOLDERS.items():
    img_dir = os.path.join(DATA_DIR, folder, "images")
    for f in os.listdir(img_dir):
        rows.append((os.path.join(img_dir, f), CLASS_NAMES.index(cls)))

df = pd.DataFrame(rows, columns=["path", "label"])
print(df.shape)
print(df["label"].value_counts().sort_index())

(21165, 2)
label
0     3616
1     6012
2    10192
3     1345
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

train: 14815 val: 3175 test: 3175


In [8]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, tf.one_hot(label, 4)

def make_dataset(dataframe, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (dataframe["path"].values, dataframe["label"].values))
    if shuffle:
        ds = ds.shuffle(len(dataframe), seed=42)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [12]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121

base = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = layers.Rescaling(1./255)(inputs)
x = layers.Normalization(mean=[0.485, 0.456, 0.406],
                         variance=[0.229**2, 0.224**2, 0.225**2])(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])
print("Model built. Total parameters:", model.count_params())

Model built. Total parameters: 7041604


In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2, 3]),
    y=train_df["label"].values)
class_weight = {i: float(w) for i, w in enumerate(weights)}
print(class_weight)

history_cw = model.fit(train_ds, validation_data=val_ds,
                       epochs=3, class_weight=class_weight)

{0: 1.4633544053733702, 1: 0.8801687262357415, 2: 0.5191687692738997, 3: 3.9317940552016983}
Epoch 1/3
463/463 ━━━━━━━━━━━━━━━━━━━━ 88s 144ms/step - accuracy: 0.6486 - loss: 0.7719 - val_accuracy: 0.7855 - val_loss: 0.5561
Epoch 2/3
463/463 ━━━━━━━━━━━━━━━━━━━━ 39s 85ms/step - accuracy: 0.7685 - loss: 0.5028 - val_accuracy: 0.8154 - val_loss: 0.4865
Epoch 3/3
463/463 ━━━━━━━━━━━━━━━━━━━━ 40s 87ms/step - accuracy: 0.7820 - loss: 0.4686 - val_accuracy: 0.8287 - val_loss: 0.4631


In [14]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = val_df["label"].values
y_prob = model.predict(val_ds)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print("Confusion matrix (rows = true class, columns = predicted):")
print(confusion_matrix(y_true, y_pred))

100/100 ━━━━━━━━━━━━━━━━━━━━ 29s 178ms/step
                 precision    recall  f1-score   support

          COVID      0.710     0.832     0.766       543
   Lung_Opacity      0.849     0.789     0.818       902
         Normal      0.876     0.832     0.853      1529
Viral_Pneumonia      0.789     0.970     0.871       201

       accuracy                          0.829      3175
      macro avg      0.806     0.856     0.827      3175
   weighted avg      0.834     0.829     0.829      3175

Confusion matrix (rows = true class, columns = predicted):
[[ 452   27   63    1]
 [  73  712  114    3]
 [ 110   99 1272   48]
 [   2    1    3  195]]


In [15]:
from tensorflow.keras import layers

# Unfreeze only the last 30 layers of DenseNet121
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
# Keep BatchNorm layers frozen (standard practice for fine-tuning)
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Recompile with a much smaller learning rate (required after changing trainable)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/lung_model.keras",
    monitor="val_loss", save_best_only=True, verbose=1)

history_ft = model.fit(train_ds, validation_data=val_ds,
                       epochs=5, class_weight=class_weight,
                       callbacks=[checkpoint])

Epoch 1/5
463/463 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.7994 - loss: 0.4293
Epoch 1: val_loss improved from None to 0.45908, saving model to /kaggle/working/lung_model.keras

Epoch 1: finished saving model to /kaggle/working/lung_model.keras
463/463 ━━━━━━━━━━━━━━━━━━━━ 94s 152ms/step - accuracy: 0.8017 - loss: 0.4208 - val_accuracy: 0.8265 - val_loss: 0.4591
Epoch 2/5
463/463 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.8062 - loss: 0.4092
Epoch 2: val_loss improved from 0.45908 to 0.42239, saving model to /kaggle/working/lung_model.keras

Epoch 2: finished saving model to /kaggle/working/lung_model.keras
463/463 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.8117 - loss: 0.4000 - val_accuracy: 0.8435 - val_loss: 0.4224
Epoch 3/5
463/463 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.8195 - loss: 0.3828
Epoch 3: val_loss improved from 0.42239 to 0.40688, saving model to /kaggle/working/lung_model.keras

Epoch 3: finished saving model to /kaggle/working/lung_model.keras

In [10]:
train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)

history = model.fit(train_ds, validation_data=val_ds, epochs=3)

Epoch 1/3
  1/463 ━━━━━━━━━━━━━━━━━━━━ 3:38:41 28s/step - accuracy: 0.4062 - loss: 1.7090

I0000 00:00:1791699234.363884     169 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


463/463 ━━━━━━━━━━━━━━━━━━━━ 122s 202ms/step - accuracy: 0.7150 - loss: 0.7383 - val_accuracy: 0.8157 - val_loss: 0.4898
Epoch 2/3
463/463 ━━━━━━━━━━━━━━━━━━━━ 38s 83ms/step - accuracy: 0.8004 - loss: 0.5236 - val_accuracy: 0.8337 - val_loss: 0.4361
Epoch 3/3
463/463 ━━━━━━━━━━━━━━━━━━━━ 39s 84ms/step - accuracy: 0.8122 - loss: 0.4912 - val_accuracy: 0.8435 - val_loss: 0.4247


In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = val_df["label"].values
y_prob = model.predict(val_ds)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print("Confusion matrix (rows = true class, columns = predicted):")
print(confusion_matrix(y_true, y_pred))

100/100 ━━━━━━━━━━━━━━━━━━━━ 29s 180ms/step
                 precision    recall  f1-score   support

          COVID      0.851     0.683     0.758       543
   Lung_Opacity      0.822     0.823     0.822       902
         Normal      0.840     0.917     0.877      1529
Viral_Pneumonia      0.982     0.811     0.888       201

       accuracy                          0.843      3175
      macro avg      0.874     0.808     0.836      3175
   weighted avg      0.845     0.843     0.842      3175

Confusion matrix (rows = true class, columns = predicted):
[[ 371   60  112    0]
 [  33  742  127    0]
 [  27   97 1402    3]
 [   5    4   29  163]]


In [1]:
print(len(train_df), len(val_df), len(test_df))
print("class_weight:", class_weight)


NameError: name 'train_df' is not defined

In [2]:
import os, numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121

DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
FOLDERS = {"COVID": "COVID", "Lung_Opacity": "Lung_Opacity", "Normal": "Normal", "Viral Pneumonia": "Viral_Pneumonia"}
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral_Pneumonia"]

rows = []
for folder, cls in FOLDERS.items():
    img_dir = os.path.join(DATA_DIR, folder, "images")
    for f in os.listdir(img_dir):
        rows.append((os.path.join(img_dir, f), CLASS_NAMES.index(cls)))
df = pd.DataFrame(rows, columns=["path", "label"])

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

IMG_SIZE, BATCH_SIZE, AUTOTUNE = 224, 32, tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, tf.one_hot(label, 4)

def make_dataset(dataframe, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((dataframe["path"].values, dataframe["label"].values))
    if shuffle:
        ds = ds.shuffle(len(dataframe), seed=42)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)

base = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False
inputs = layers.Input(shape=(224, 224, 3))
x = layers.Rescaling(1./255)(inputs)
x = layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2])(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation="softmax")(x)
model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1, 2, 3]), y=train_df["label"].values)
class_weight = {i: float(w) for i, w in enumerate(weights)}
history_cw = model.fit(train_ds, validation_data=val_ds, epochs=3, class_weight=class_weight)

I0000 00:00:1791710332.484203      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791710332.487480      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/3
  1/463 ━━━━━━━━━━━━━━━━━━━━ 3:23:16 26s/step - accuracy: 0.4375 - loss: 2.5722

I0000 00:00:1791710365.991957     148 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


463/463 ━━━━━━━━━━━━━━━━━━━━ 103s 166ms/step - accuracy: 0.6533 - loss: 0.7572 - val_accuracy: 0.7767 - val_loss: 0.5801
Epoch 2/3
463/463 ━━━━━━━━━━━━━━━━━━━━ 42s 91ms/step - accuracy: 0.7586 - loss: 0.5104 - val_accuracy: 0.8072 - val_loss: 0.5154
Epoch 3/3
463/463 ━━━━━━━━━━━━━━━━━━━━ 42s 90ms/step - accuracy: 0.7811 - loss: 0.4650 - val_accuracy: 0.8167 - val_loss: 0.4853


In [3]:
# Unfreeze only the last 30 layers of DenseNet121
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
# Keep BatchNorm layers frozen (standard practice for fine-tuning)
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Recompile with a much smaller learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/lung_model.keras",
    monitor="val_loss", save_best_only=True, verbose=1)

history_ft = model.fit(train_ds, validation_data=val_ds,
                       epochs=5, class_weight=class_weight,
                       callbacks=[checkpoint])

Epoch 1/5
463/463 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.8003 - loss: 0.4348
Epoch 1: val_loss improved from None to 0.43545, saving model to /kaggle/working/lung_model.keras

Epoch 1: finished saving model to /kaggle/working/lung_model.keras
463/463 ━━━━━━━━━━━━━━━━━━━━ 98s 164ms/step - accuracy: 0.7997 - loss: 0.4248 - val_accuracy: 0.8387 - val_loss: 0.4354
Epoch 2/5
463/463 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.8124 - loss: 0.4138
Epoch 2: val_loss improved from 0.43545 to 0.42526, saving model to /kaggle/working/lung_model.keras

Epoch 2: finished saving model to /kaggle/working/lung_model.keras
463/463 ━━━━━━━━━━━━━━━━━━━━ 45s 98ms/step - accuracy: 0.8148 - loss: 0.4058 - val_accuracy: 0.8425 - val_loss: 0.4253
Epoch 3/5
463/463 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.8193 - loss: 0.3866
Epoch 3: val_loss improved from 0.42526 to 0.41047, saving model to /kaggle/working/lung_model.keras

Epoch 3: finished saving model to /kaggle/working/lung_model.kera

In [4]:
import os
for f in os.listdir("/kaggle/working"):
    print(f, round(os.path.getsize(os.path.join("/kaggle/working", f)) / 1e6, 1), "MB")

.virtual_documents 0.0 MB
lung_model.keras 34.8 MB


In [5]:
import os, glob
print(os.listdir("/kaggle/working"))
print(glob.glob("/kaggle/input/**/lung_model.keras", recursive=True))

['.virtual_documents', 'lung_model.keras']
[]


In [6]:
import os, numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

MODEL_PATH = "/kaggle/working/lung_model.keras"

DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
FOLDERS = {"COVID": "COVID", "Lung_Opacity": "Lung_Opacity", "Normal": "Normal", "Viral Pneumonia": "Viral_Pneumonia"}
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral_Pneumonia"]

rows = []
for folder, cls in FOLDERS.items():
    img_dir = os.path.join(DATA_DIR, folder, "images")
    for f in os.listdir(img_dir):
        rows.append((os.path.join(img_dir, f), CLASS_NAMES.index(cls)))
df = pd.DataFrame(rows, columns=["path", "label"])

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)
print("test images:", len(test_df))

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    return img, tf.one_hot(label, 4)

test_ds = (tf.data.Dataset.from_tensor_slices((test_df["path"].values, test_df["label"].values))
           .map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE))

best = tf.keras.models.load_model(MODEL_PATH)
y_true = test_df["label"].values
y_prob = best.predict(test_ds)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print(confusion_matrix(y_true, y_pred))

out = pd.DataFrame({"path": test_df["path"].values, "true_label": y_true, "pred_label": y_pred})
for i, name in enumerate(CLASS_NAMES):
    out["prob_" + name] = y_prob[:, i]
out.to_csv("/kaggle/working/predictions.csv", index=False)
print("saved predictions.csv")

test images: 3175
100/100 ━━━━━━━━━━━━━━━━━━━━ 30s 191ms/step
                 precision    recall  f1-score   support

          COVID      0.812     0.887     0.848       542
   Lung_Opacity      0.853     0.854     0.853       902
         Normal      0.915     0.870     0.892      1529
Viral_Pneumonia      0.868     0.975     0.918       202

       accuracy                          0.875      3175
      macro avg      0.862     0.897     0.878      3175
   weighted avg      0.877     0.875     0.875      3175

[[ 481   26   32    3]
 [  43  770   88    1]
 [  67  106 1330   26]
 [   1    1    3  197]]
saved predictions.csv
